READ SILVER_SALES

In [1]:
# 1. Required libraries

import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os

# Load environment variable

load_dotenv()

mysql_user = os.getenv("MYSQL_USER")
mysql_password = quote_plus(os.getenv("MYSQL_PASSWORD"))
mysql_host = os.getenv("MYSQL_HOST")
mysql_port = os.getenv("MYSQL_PORT")
mysql_database = os.getenv("MYSQL_DATABASE")

In [2]:
# 2. CONNECT MYSQL 

try:
    engine = create_engine(
        f"mysql+pymysql://{mysql_user}:{mysql_password}@"
        f"{mysql_host}:{mysql_port}/{mysql_database}"
        
    )

    # test connection
    with engine.connect() as connection:
        print("Successfully connected MYSQL !")


except Exception as e:
    print(f"Error: Couldn't connect to MYSQL {e}")

Successfully connected MYSQL !


In [4]:
# 3. READ SILVER

df = pd.read_sql(
    "SELECT * FROM silver_sales",
    engine
)

print("Silver data loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Silver data loaded successfully!
Rows: 9800
Columns: 22


4. CREATE THE DIMENSION DATAFRAMES

In [5]:
# 4.1 CUSTOMER DIMENSION

dim_customer = df[
    [
        "customer_id",
        "customer_name",
        "segment"
    ]
].drop_duplicates()

print("Customer dimension rows:", len(dim_customer))

Customer dimension rows: 793


In [7]:
# 4.2 PRODUCT DIMENSION

dim_product = df[
    [
        "product_id",
        "product_name",
        "category",
        "sub_category"
    ]
].drop_duplicates()

print("Product dimension rows:", len(dim_product))

Product dimension rows: 1893


In [9]:
# 4.3 LOCATION DIMENSION

dim_location = df[
    [
        "country",
        "city",
        "state",
        "postal_code",
        "region"
    ]
].drop_duplicates()

print("Location dimension rows:", len(dim_location))

Location dimension rows: 628


In [10]:
# 4.4 DATE DIMENSION

dim_date = df[
    [
        "order_date"
    ]
].drop_duplicates()

print("Date dimension rows:", len(dim_date))

Date dimension rows: 1230


In [17]:
# Convert the column to datetime first
dim_date["order_date"] = pd.to_datetime(dim_date["order_date"])

dim_date["year"] = dim_date["order_date"].dt.year
dim_date["month"] = dim_date["order_date"].dt.month
dim_date["month_name"] = dim_date["order_date"].dt.month_name()
dim_date["quarter"] = dim_date["order_date"].dt.quarter
dim_date["day"] = dim_date["order_date"].dt.day
dim_date["day_name"] = dim_date["order_date"].dt.day_name()

print(dim_date)

print(dim_date["order_date"].dtype)

     order_date  year  month month_name  quarter  day   day_name
0    2017-11-08  2017     11   November        4    8  Wednesday
2    2017-06-12  2017      6       June        2   12     Monday
3    2016-10-11  2016     10    October        4   11    Tuesday
5    2015-06-09  2015      6       June        2    9    Tuesday
12   2018-04-15  2018      4      April        2   15     Sunday
...         ...   ...    ...        ...      ...  ...        ...
9696 2015-06-10  2015      6       June        2   10  Wednesday
9750 2017-10-11  2017     10    October        4   11  Wednesday
9764 2015-06-18  2015      6       June        2   18   Thursday
9765 2018-02-28  2018      2   February        1   28  Wednesday
9785 2016-05-09  2016      5        May        2    9     Monday

[1230 rows x 7 columns]
datetime64[ns]


In [25]:
# surrogate/date key

# rename 

dim_date = dim_date.rename(
    columns = {"order_date": "full_date"}
)

print(dim_date.dtypes)

dim_date["date_key"] = (
    dim_date["full_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

print(dim_date.columns)

full_date     datetime64[ns]
year                   int32
month                  int32
month_name            object
quarter                int32
day                    int32
day_name              object
date_key               int64
dtype: object
Index(['full_date', 'year', 'month', 'month_name', 'quarter', 'day',
       'day_name', 'date_key'],
      dtype='object')


In [27]:
print(dim_date.head())
print(dim_date.columns)

print(dim_date.shape)
print(dim_date["date_key"].is_unique)

    full_date  year  month month_name  quarter  day   day_name  date_key
0  2017-11-08  2017     11   November        4    8  Wednesday  20171108
2  2017-06-12  2017      6       June        2   12     Monday  20170612
3  2016-10-11  2016     10    October        4   11    Tuesday  20161011
5  2015-06-09  2015      6       June        2    9    Tuesday  20150609
12 2018-04-15  2018      4      April        2   15     Sunday  20180415
Index(['full_date', 'year', 'month', 'month_name', 'quarter', 'day',
       'day_name', 'date_key'],
      dtype='object')
(1230, 8)
True


5. Inspection of DIMENSION 

5.1 DIM_CUSTOMER

In [28]:
print("========== DIM CUSTOMER ==========")

print("Rows:", len(dim_customer))
print("Columns:", len(dim_customer.columns))

print("\nColumns:")
print(dim_customer.columns.tolist())

print("\nFirst 5 rows:")
print(dim_customer.head())

========== DIM CUSTOMER ==========
Rows: 793
Columns: 3

Columns:
['customer_id', 'customer_name', 'segment']

First 5 rows:
   customer_id    customer_name    segment
0     CG-12520      Claire Gute   Consumer
2     DV-13045  Darrin Van Huff  Corporate
3     SO-20335   Sean O'Donnell   Consumer
5     BH-11710  Brosina Hoffman   Consumer
12    AA-10480     Andrew Allen   Consumer


In [29]:
print("\nDuplicate customer_ids:",
      dim_customer["customer_id"].duplicated().sum())

print("Unique customer_ids:",
      dim_customer["customer_id"].nunique())


Duplicate customer_ids: 0
Unique customer_ids: 793


In [30]:
assert dim_customer["customer_id"].nunique() == len(dim_customer)

In [33]:
print("\nMissing values:")
print(dim_customer.isna().sum())

assert dim_customer["customer_id"].notna().all()
assert dim_customer["customer_name"].notna().all()
assert dim_customer["segment"].notna().all()

print("Customer dimension validation passed!")


Missing values:
customer_id      0
customer_name    0
segment          0
dtype: int64
Customer dimension validation passed!


5.2 DIM PRODUCT

In [35]:
print("\n========== DIM PRODUCT ==========")

print("Rows:", len(dim_product))
print("Columns:", len(dim_product.columns))

print("\nColumns:")
print(dim_product.columns.tolist())

print("\nFirst 5 rows:")
print(dim_product.head())


========== DIM PRODUCT ==========
Rows: 1893
Columns: 4

Columns:
['product_id', 'product_name', 'category', 'sub_category']

First 5 rows:
        product_id                                       product_name  \
0  FUR-BO-10001798                  Bush Somerset Collection Bookcase   
1  FUR-CH-10000454  Hon Deluxe Fabric Upholstered Stacking Chairs,...   
2  OFF-LA-10000240  Self-Adhesive Address Labels for Typewriters b...   
3  FUR-TA-10000577      Bretford CR4500 Series Slim Rectangular Table   
4  OFF-ST-10000760                     Eldon Fold 'N Roll Cart System   

          category sub_category  
0        Furniture    Bookcases  
1        Furniture       Chairs  
2  Office Supplies       Labels  
3        Furniture       Tables  
4  Office Supplies      Storage  


In [36]:
print("\nDuplicate product_ids:",
      dim_product["product_id"].duplicated().sum())

print("Unique product_ids:",
      dim_product["product_id"].nunique())


Duplicate product_ids: 32
Unique product_ids: 1861


In [ ]:
duplicate_product_ids = (
    dim_product[
        dim_product["product_id"].duplicated(keep=False)
    ]
    .sort_values("product_id")
)

print(duplicate_product_ids) 

           product_id                                       product_name  \
2471  FUR-BO-10002213   Sauder Forest Hills Library, Woodland Oak Finish   
2115  FUR-BO-10002213              DMI Eclipse Executive Suite Bookcases   
66    FUR-CH-10001146        Global Value Mid-Back Manager's Chair, Gray   
128   FUR-CH-10001146                           Global Task Chair, Black   
1459  FUR-FU-10001473                            DAX Wood Document Frame   
...               ...                                                ...   
1219  TEC-PH-10002200                              Samsung Galaxy Note 2   
2596  TEC-PH-10002310  Plantronics Calisto P620-M USB Wireless Speake...   
1378  TEC-PH-10002310                 Panasonic KX T7731-B Digital phone   
922   TEC-PH-10004531      OtterBox Commuter Series Case - iPhone 5 & 5s   
2713  TEC-PH-10004531                                        AT&T CL2909   

        category sub_category  
2471   Furniture    Bookcases  
2115   Furniture    Boo

In [40]:
print(
    duplicate_product_ids["product_id"]
    .value_counts()
)

product_id
FUR-BO-10002213    2
FUR-CH-10001146    2
TEC-PH-10002310    2
TEC-PH-10002200    2
TEC-PH-10001795    2
TEC-PH-10001530    2
TEC-MA-10001148    2
TEC-AC-10003832    2
TEC-AC-10002550    2
TEC-AC-10002049    2
OFF-ST-10004950    2
OFF-ST-10001228    2
OFF-PA-10003022    2
OFF-PA-10002377    2
OFF-PA-10002195    2
OFF-PA-10001970    2
OFF-PA-10001166    2
OFF-PA-10000659    2
OFF-PA-10000477    2
OFF-PA-10000357    2
OFF-BI-10004654    2
OFF-BI-10004632    2
OFF-BI-10002026    2
OFF-AR-10001149    2
OFF-AP-10000576    2
FUR-FU-10004864    2
FUR-FU-10004848    2
FUR-FU-10004270    2
FUR-FU-10004091    2
FUR-FU-10004017    2
FUR-FU-10001473    2
TEC-PH-10004531    2
Name: count, dtype: int64


In [42]:
duplicate_products = (
    dim_product[
        dim_product["product_id"].duplicated(keep=False)
    ]
    .sort_values("product_id")
)

print(
    duplicate_products[
        ["product_id", "product_name", "category", "sub_category"]
    ].to_string(index=False)
)

     product_id                                                                                    product_name        category sub_category
FUR-BO-10002213                                                Sauder Forest Hills Library, Woodland Oak Finish       Furniture    Bookcases
FUR-BO-10002213                                                           DMI Eclipse Executive Suite Bookcases       Furniture    Bookcases
FUR-CH-10001146                                                     Global Value Mid-Back Manager's Chair, Gray       Furniture       Chairs
FUR-CH-10001146                                                                        Global Task Chair, Black       Furniture       Chairs
FUR-FU-10001473                                                                         DAX Wood Document Frame       Furniture  Furnishings
FUR-FU-10001473                                          Eldon Executive Woodline II Desk Accessories, Mahogany       Furniture  Furnishings
FUR-FU-100040

In [44]:
conflicting_products = (
    dim_product
    .groupby("product_id")
    .agg(
        product_name_count=("product_name", "nunique"),
        category_count=("category", "nunique"),
        sub_category_count=("sub_category", "nunique")
    )
)

print(
    conflicting_products[
        (conflicting_products["product_name_count"] > 1) |
        (conflicting_products["category_count"] > 1) |
        (conflicting_products["sub_category_count"] > 1)
    ]
)

                 product_name_count  category_count  sub_category_count
product_id                                                             
FUR-BO-10002213                   2               1                   1
FUR-CH-10001146                   2               1                   1
FUR-FU-10001473                   2               1                   1
FUR-FU-10004017                   2               1                   1
FUR-FU-10004091                   2               1                   1
FUR-FU-10004270                   2               1                   1
FUR-FU-10004848                   2               1                   1
FUR-FU-10004864                   2               1                   1
OFF-AP-10000576                   2               1                   1
OFF-AR-10001149                   2               1                   1
OFF-BI-10002026                   2               1                   1
OFF-BI-10004632                   2               1             

Multiple product names for same `product_id` |

32 product IDs have 2 distinct product names | 

Product dimension should have one record per product ID |
 
Investigate source inconsistency; preserve source records until business rule is established |


In [46]:
print("\nMissing values:")
print(dim_product.isna().sum())


Missing values:
product_id      0
product_name    0
category        0
sub_category    0
dtype: int64


In [47]:
assert dim_product["product_id"].notna().all()
assert dim_product["product_name"].notna().all()
assert dim_product["category"].notna().all()
assert dim_product["sub_category"].notna().all()

print("\nDistinct product records:", len(dim_product))

print("Product dimension validation passed!")


Distinct product records: 1893
Product dimension validation passed!


In [ ]:
# explicitly detecting and documenting the anology 

conflicting_product_ids = (
    dim_product
    .groupby("product_id")["product_name"]
    .nunique()
)

conflicting_product_ids = conflicting_product_ids[
    conflicting_product_ids > 1
]

print("\nProduct IDs with multiple product names:",
      len(conflicting_product_ids))

assert len(conflicting_product_ids) == 32


Product IDs with multiple product names: 32


5.3 DIMENSION LOCATION

In [49]:
print("\n========== DIM LOCATION ==========")

print("Rows:", len(dim_location))
print("Columns:", len(dim_location.columns))

print("\nColumns:")
print(dim_location.columns.tolist())

print("\nFirst 5 rows:")
print(dim_location.head())


========== DIM LOCATION ==========
Rows: 628
Columns: 5

Columns:
['country', 'city', 'state', 'postal_code', 'region']

First 5 rows:
          country             city           state postal_code region
0   United States        Henderson        Kentucky       42420  South
2   United States      Los Angeles      California       90036   West
3   United States  Fort Lauderdale         Florida       33311  South
5   United States      Los Angeles      California       90032   West
12  United States          Concord  North Carolina       28027  South


In [50]:
# check duplicates with all five attributes

location_columns = [
    "country",
    "city",
    "state",
    "postal_code",
    "region"
]

duplicate_locations = dim_location.duplicated(
    subset=location_columns
).sum()

print("Duplicate locations:", duplicate_locations)

Duplicate locations: 0


In [52]:
assert duplicate_locations == 0

In [53]:
# check nulls

print("\nMissing values:")
print(dim_location.isna().sum())


Missing values:
country        0
city           0
state          0
postal_code    1
region         0
dtype: int64


In [54]:
print(
    "Missing postal codes:",
    dim_location["postal_code"].isna().sum()
)

Missing postal codes: 1


In [55]:
assert dim_location["country"].notna().all()
assert dim_location["city"].notna().all()
assert dim_location["state"].notna().all()
assert dim_location["region"].notna().all()

print("Location dimension validation passed!")

Location dimension validation passed!


5.4 DIMENSION DATE

In [56]:
print("\n========== DIM DATE ==========")

print("Rows:", len(dim_date))
print("Columns:", len(dim_date.columns))

print("\nColumns:")
print(dim_date.columns.tolist())

print("\nFirst 5 rows:")
print(dim_date.head())


========== DIM DATE ==========
Rows: 1230
Columns: 8

Columns:
['full_date', 'year', 'month', 'month_name', 'quarter', 'day', 'day_name', 'date_key']

First 5 rows:
    full_date  year  month month_name  quarter  day   day_name  date_key
0  2017-11-08  2017     11   November        4    8  Wednesday  20171108
2  2017-06-12  2017      6       June        2   12     Monday  20170612
3  2016-10-11  2016     10    October        4   11    Tuesday  20161011
5  2015-06-09  2015      6       June        2    9    Tuesday  20150609
12 2018-04-15  2018      4      April        2   15     Sunday  20180415


In [57]:
# Check unique dates

print("\nDuplicate date_keys:",
      dim_date["date_key"].duplicated().sum())

print("Unique date_keys:",
      dim_date["date_key"].nunique())


Duplicate date_keys: 0
Unique date_keys: 1230


In [58]:
assert dim_date["date_key"].nunique() == len(dim_date)

In [59]:
# check full date uniqueness

print(
    "Duplicate full_dates:",
    dim_date["full_date"].duplicated().sum()
)

Duplicate full_dates: 0


In [60]:
assert dim_date["full_date"].nunique() == len(dim_date)

In [61]:
print("\nMissing values:")
print(dim_date.isna().sum())


Missing values:
full_date     0
year          0
month         0
month_name    0
quarter       0
day           0
day_name      0
date_key      0
dtype: int64


In [62]:
assert dim_date.isna().sum().sum() == 0

In [64]:
expected_date_keys = (
    dim_date["full_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

assert (
    dim_date["date_key"] == expected_date_keys
).all()

print("Date key consistency passed!")

Date key consistency passed!


FINAL VALIDATION CHECK

In [65]:
print("\n========================================")
print("       GOLD DIMENSION VALIDATION")
print("========================================")

print(f"Customer rows : {len(dim_customer)}")
print(f"Product rows  : {len(dim_product)}")
print(f"Location rows : {len(dim_location)}")
print(f"Date rows     : {len(dim_date)}")

print("\nCustomer unique IDs:",
      dim_customer["customer_id"].nunique())

print("Product unique IDs:",
      dim_product["product_id"].nunique())

print("Duplicate locations:",
      dim_location.duplicated(
          subset=location_columns
      ).sum())

print("Date unique keys:",
      dim_date["date_key"].nunique())

print("\nAll Gold dimension validations passed!")


       GOLD DIMENSION VALIDATION
Customer rows : 793
Product rows  : 1893
Location rows : 628
Date rows     : 1230

Customer unique IDs: 793
Product unique IDs: 1861
Duplicate locations: 0
Date unique keys: 1230

All Gold dimension validations passed!
